# Lab 15 — Performance Investigation

**Tujuan:** Menggunakan Spark UI untuk menilai shuffle, partisi, dan query plan sebelum mengubah kode.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 15")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

## 1. Starting workload

Dataset 2 juta baris dipilih agar sesuai worker 2 GB; gunakan UI bukan stopwatch saja.

In [ ]:
df = spark.range(0, 2_000_000, 1, 4)
data = df.withColumn("group_id", F.col("id") % 100)
heavy = data.repartition(20)
result = heavy.groupBy("group_id").count()
result.explain("formatted")

In [ ]:
from time import perf_counter
start = perf_counter()
result.collect()
print("Elapsed sec:", round(perf_counter() - start, 2))

## 2. Change one variable

In [ ]:
smaller = data.repartition(4)
result_4 = smaller.groupBy("group_id").count()
result_4.explain("formatted")
start = perf_counter()
result_4.collect()
print("Elapsed sec:", round(perf_counter() - start, 2))

## 3. Try excessive partitions

In [ ]:
larger = data.repartition(100)
result_100 = larger.groupBy("group_id").count()
result_100.explain("formatted")
start = perf_counter()
result_100.collect()
print("Elapsed sec:", round(perf_counter() - start, 2))

## Investigation sheet

Untuk masing-masing percobaan catat **stages**, **tasks/stage**, **shuffle read/write**, **executor runtime**, **GC time**, **spilled bytes**, dan durasi. Spark AQE dapat menyesuaikan jumlah shuffle partitions sehingga hasil aktual dapat berbeda dari angka yang ditulis.

**Tugas akhir:** buat rekomendasi berdasarkan Spark UI dan `explain`, bukan asumsi bahwa 100 partisi pasti lebih cepat dari 4.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.